# H Final Test: Frozen H3 Deep Feature Fusion

**Project:** Stain-Robust Explainable Deep Feature Fusion Framework for Lung Cancer Histopathology Subtype Classification  
**Selected image checkpoint:** `H3_MobileNetV2_EfficientNetB0_fusion_seed42/fusion_image_model.keras`  
**Partition:** Original fixed **held-out TEST**: 2,333 lung images from 113 reconstructed source-image groups  
**Purpose:** One final evaluation of the model selected using validation before opening test images.

## Execution and preservation policy

- **Run one numbered section at a time**, review output, and do not select *Run all*.
- Sections 1, 2, 4, and 5 access only saved artifacts or test **metadata**. Section 3 reads three **validation** images to confirm saved H3 inference. No test image pixels are read until Section 6.
- Section 5 requires an exact human authorization phrase. Section 6 is the **one planned test inference**, with a fresh permanent Drive attempt record written before first test-pixel decode. If interrupted, stop and preserve the partial CSV. Do not automatically rerun test inference.
- Section 7 reports predetermined metrics using saved predictions only. Section 8 verifies artifact integrity read-only.
- **No training, resplitting, augmentation, stain normalization, model selection, threshold tuning, or changes to source checkpoints.** No H4 stain transformation belongs in this selected H3 pipeline.
- The H1 MobileNetV2 and H2 EfficientNetB0 pooled features (1,280 values each) feed the original H3 2,560-dimensional fused head. The exported H3 full image model is used directly for final inference with its internal backbone preprocessing. Input RGB pixels are resized to 224 x 224 using bilinear padding and antialiasing, then supplied as float32 in [0,255].
- Results are stored **only** under the lung project, separate from ultrasound and previous H1/H2/H3/H4 results. Original checkpoints, fixed split manifests, model-selection freeze, and historical validation records are immutable.

## Prespecified metrics and interpretation

Report TEST accuracy, macro-precision, macro-recall, macro-F1, per-class precision/recall/F1/support, and a 3 x 3 confusion matrix using the class order `lung_aca`, `lung_n`, `lung_scc`. Keep this held-out TEST separate from the previously recorded H1/H2/H3/H4 **validation** results. Lung source-image groups are **not** patient or hospital identifiers; these metrics are not external-hospital performance or clinical diagnosis evidence.

## Section 1: Verify H3 freeze and immutable source checkpoints (no test pixels)

Connect Google Drive, verify the frozen H3 full-image model and head, the original H1/H2 reference checkpoints, configuration, split manifest and class mapping against the saved freeze and prior SHA-256 records. Detect any earlier final-test attempt and refuse an uncontrolled repeat. No test images are decoded.

In [1]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import platform
import shutil
import stat
import zipfile

from google.colab import drive
import tensorflow as tf
from tensorflow import keras

ROOT = Path('/content/drive/MyDrive/ICTIU_Final_Projects')
if not ROOT.is_dir():
    drive.mount('/content/drive')
if not ROOT.is_dir():
    raise FileNotFoundError('Google Drive project root is unavailable.')

PROJECT = ROOT / 'lung'
H1_RUN = PROJECT / 'results/H1_MobileNetV2_frozen_seed42'
H2_RUN = PROJECT / 'results/H2_EfficientNetB0_frozen_seed42'
H3_RUN = PROJECT / 'results/H3_MobileNetV2_EfficientNetB0_fusion_seed42'
H1_MODEL = H1_RUN / 'best_model.keras'
H2_MODEL = H2_RUN / 'best_model.keras'
H3_MODEL = H3_RUN / 'fusion_image_model.keras'
H3_HEAD = H3_RUN / 'best_head.keras'
H3_CONFIG = H3_RUN / 'configuration.json'
H3_VAL = H3_RUN / 'validation_predictions.csv'
MANIFEST_PATH = PROJECT / 'splits/manifest_candidate.csv'
MAPPING_PATH = PROJECT / 'splits/class_mapping.json'
AUDIT_PATH = PROJECT / 'results/audit.json'
ZIP_PATH = PROJECT / 'archives/LC25000.zip'
FREEZE_PATH = PROJECT / 'selection/H3_model_selection_freeze_2026-10-19.json'
PROTOCOL_PATH = PROJECT / 'selection/H3_final_test_protocol_2026-10-19.json'
RUN_NAME = 'H_Final_Test_H3_seed42'
RESULT_DIR = PROJECT / 'results' / RUN_NAME
RAW_PRED_PATH = RESULT_DIR / 'test_predictions.csv'
STARTED_PATH = RESULT_DIR / 'test_evaluation_started.json'
COMPLETE_PATH = RESULT_DIR / 'test_inference_completed.json'
FINAL_MARKER = RESULT_DIR / 'FINALIZED.json'

EXPECTED_SHA = {
    'H1_model': '503559b9333124d8be0b665a9fd9538220a750db2274ca55a9e61dff503dd6bf',
    'H2_model': 'fcd95f3198c1a63641462b3a72e6b21cf501a4ad0928186bd91c2362c53fa5f6',
    'H3_model': '9e1a5bbf49da7e88a529f5ac08303c65e138d7c9f818959aaddd46f609890c95',
    'H3_head': '77deea98faef81db0fd8e6574598f677a18f9368fde3edccc67c3549b5098b75',
    'H3_config': 'efac45117571f4223de67c62dfcfd4751ba52122d972885d19222782cf9f386c',
    'manifest': 'd931c9b11d03716c3c426f251bc75eaa2051002acca99d3ba66d59d964285dea',
    'mapping': '4d14782118e3a438c13b209d7a594f2f9676d09cec25125b599d8c6d6c26a6a8',
}
EXPECTED_ARCHIVE_SHA256 = '382c98315435cbbf559be5da94cc0b62f696434e30e15feefebe9c892de01728'
FILES_TO_CHECK = {
    'H1_model': H1_MODEL, 'H2_model': H2_MODEL,
    'H3_model': H3_MODEL, 'H3_head': H3_HEAD, 'H3_config': H3_CONFIG,
    'manifest': MANIFEST_PATH, 'mapping': MAPPING_PATH,
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

required = list(FILES_TO_CHECK.values()) + [H3_VAL, AUDIT_PATH, ZIP_PATH, FREEZE_PATH]
missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError('Required evidence missing: ' + str(missing))

TEST_RUN_STATE = 'NOT_STARTED'
if RESULT_DIR.exists():
    if (RESULT_DIR / 'FINALIZED.json').is_file():
        TEST_RUN_STATE = 'FINALIZED'
    elif (RESULT_DIR / 'test_inference_completed.json').is_file():
        TEST_RUN_STATE = 'INFERENCE_COMPLETED'
    else:
        raise RuntimeError('Previous incomplete TEST attempt exists. Stop for controlled recovery: '
                           + str(RESULT_DIR))

for key, path in FILES_TO_CHECK.items():
    if sha256_file(path) != EXPECTED_SHA[key]:
        raise RuntimeError('Frozen artifact checksum mismatch: ' + key)
freeze = read_json(FREEZE_PATH)
assert freeze['decision_status'] == 'FROZEN_BEFORE_HELD_OUT_TEST'
assert freeze['selected_experiment'] == 'H3_MobileNetV2_EfficientNetB0_fusion_seed42'
assert freeze['expected_test_images'] == 2333
assert freeze['final_test_protocol']['status_at_freeze'] == 'NOT_EVALUATED'
assert freeze['final_test_protocol']['planned_evaluations'] == 1
assert freeze['final_test_protocol']['test_based_tuning_allowed'] is False
for claim, path in [
    ('model_artifact', H3_MODEL), ('head_artifact', H3_HEAD),
    ('configuration', H3_CONFIG), ('manifest', MANIFEST_PATH),
    ('mapping_file', MAPPING_PATH),
]:
    entry = freeze[claim]
    assert ROOT / entry['path_relative_to_root'] == path, claim
    assert sha256_file(path) == entry['sha256'], claim

H3_CONFIGURATION = read_json(H3_CONFIG)
assert H3_CONFIGURATION['run_name'] == 'H3_MobileNetV2_EfficientNetB0_fusion_seed42'
assert H3_CONFIGURATION['source_sha256']['manifest'] == EXPECTED_SHA['manifest']
assert H3_CONFIGURATION['archive_sha256'] == EXPECTED_ARCHIVE_SHA256
assert H3_CONFIGURATION['reference_checkpoint_sha256'] == {
    'H1': EXPECTED_SHA['H1_model'], 'H2': EXPECTED_SHA['H2_model']}
assert H3_CONFIGURATION['backbones_frozen'] is True
assert H3_CONFIGURATION['stain_normalization'] is False
assert H3_CONFIGURATION['stain_aware_augmentation'] is False
assert H3_CONFIGURATION['feature_fusion'] is True
assert H3_CONFIGURATION['held_out_test_evaluated'] is False

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Frozen H3 full model SHA-256:', EXPECTED_SHA['H3_model'])
print('Frozen H3 head SHA-256:', EXPECTED_SHA['H3_head'])
print('H1/H2 checkpoint provenance: verified')
print('Selection freeze:', FREEZE_PATH.name)
print('Previous final TEST run state:', TEST_RUN_STATE)
if PROTOCOL_PATH.exists():
    print('Existing TEST protocol detected; Section 4 will verify it read-only.')
print('SECTION 1 PASS: Frozen H3 source artifacts match. No TEST pixels read.')

Mounted at /content/drive
Python: 3.13.15
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Frozen H3 full model SHA-256: 9e1a5bbf49da7e88a529f5ac08303c65e138d7c9f818959aaddd46f609890c95
Frozen H3 head SHA-256: 77deea98faef81db0fd8e6574598f677a18f9368fde3edccc67c3549b5098b75
H1/H2 checkpoint provenance: verified
Selection freeze: H3_model_selection_freeze_2026-10-19.json
Previous final TEST run state: NOT_STARTED
SECTION 1 PASS: Frozen H3 source artifacts match. No TEST pixels read.


## Section 2: Audit original held-out source-group split (metadata only)

Inspect the unchanged lung manifest without opening test images. Preserve its original row order, three fixed class IDs, 15,000 image assignments and 740 reconstructed source-image groups. The 2,333 test images belong to 113 reconstructed source-image groups, **not 113 patients**. Compare per-class test counts to the original handoff, and seal the test row order using SHA-256.

In [2]:
import pandas as pd

EXPECTED_CLASSES = {'lung_aca': 0, 'lung_n': 1, 'lung_scc': 2}
class_to_id = read_json(MAPPING_PATH)
assert class_to_id == freeze['class_to_id'] == EXPECTED_CLASSES
assert H3_CONFIGURATION['class_to_id'] == EXPECTED_CLASSES
CLASS_NAMES = [name for name, _ in sorted(class_to_id.items(), key=lambda p: p[1])]
raw = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)

ALIASES = {
    'path': ['relative_path', 'rel_path', 'relpath', 'image_path',
             'filepath', 'file_path', 'path', 'filename', 'image_name'],
    'split': ['split', 'subset', 'partition'],
    'group': ['group_id', 'source_group_id', 'source_group',
              'source_image_group', 'original_group', 'source_image_id'],
    'name': ['class_name', 'category', 'class', 'label_name'],
    'id': ['class_id', 'label_id', 'label_idx'],
    'hash': ['sha256', 'file_sha256', 'image_sha256', 'sha256_hash'],
}
lookup = {str(c).strip().lower(): c for c in raw.columns}

def column(role, required=True):
    found = next((lookup[k] for k in ALIASES[role] if k in lookup), None)
    if required and found is None:
        raise ValueError('Missing manifest field: ' + role)
    return found

COL = {role: column(role, required=(role != 'name')) for role in ALIASES}
if COL['name'] is None and COL['id'] is None:
    raise ValueError('Manifest must contain class names or class IDs.')

splits = raw[COL['split']].str.strip().str.lower().map({
    'train': 'train', 'training': 'train', 'val': 'val',
    'valid': 'val', 'validation': 'val', 'test': 'test', 'testing': 'test'})
if splits.isna().any():
    raise ValueError('Unrecognized split values in the original manifest.')

if COL['name'] is not None:
    names = raw[COL['name']].str.strip()
    if not names.isin(EXPECTED_CLASSES).all():
        raise ValueError('Unexpected class name in manifest.')
    ids = names.map(class_to_id).astype(int)
else:
    ids = raw[COL['id']].astype(int)
    names = ids.map({v: k for k, v in class_to_id.items()})
if COL['id'] is not None and not (raw[COL['id']].astype(int) == ids).all():
    raise ValueError('Manifest class names and IDs disagree.')

manifest = pd.DataFrame({
    'source_path': raw[COL['path']].str.strip(),
    'split': splits,
    'source_group': raw[COL['group']].str.strip(),
    'class_id': ids,
    'class_name': names,
    'recorded_sha256': raw[COL['hash']].str.strip().str.lower(),
})
assert len(manifest) == 15000
assert manifest['source_path'].is_unique
assert not manifest[['source_path', 'source_group', 'class_name',
                     'recorded_sha256']].eq('').any().any()
assert manifest['recorded_sha256'].str.fullmatch(r'[0-9a-f]{64}').all()
assert manifest['split'].value_counts().to_dict() == {
    'train': 10394, 'val': 2273, 'test': 2333}
assert manifest.groupby('split')['source_group'].nunique().to_dict() == {
    'train': 514, 'val': 113, 'test': 113}
assert manifest['source_group'].nunique() == 740
assert (manifest.groupby('source_group')['split'].nunique() == 1).all()
assert (manifest.groupby('source_group')['class_id'].nunique() == 1).all()
assert (manifest.groupby('recorded_sha256')['split'].nunique() == 1).all()
assert manifest['class_name'].value_counts().to_dict() == {k: 5000 for k in EXPECTED_CLASSES}

TEST_ROWS = manifest.loc[manifest['split'] == 'test'].copy().reset_index(drop=True)
VAL_ROWS = manifest.loc[manifest['split'] == 'val'].copy().reset_index(drop=True)
assert TEST_ROWS['class_id'].value_counts().sort_index().to_dict() == {0: 768, 1: 788, 2: 777}
assert TEST_ROWS['source_group'].nunique() == 113
assert not set(TEST_ROWS['source_group']).intersection(set(VAL_ROWS['source_group']))
assert len(TEST_ROWS) == 2333
TEST_ORDER_SHA = hashlib.sha256(
    '\n'.join(TEST_ROWS['source_path'].tolist()).encode('utf-8')
).hexdigest()
print('Original lung manifest rows:', len(manifest))
print('Held-out TEST images:', len(TEST_ROWS))
print('Reconstructed TEST source-image groups (NOT patients):', TEST_ROWS['source_group'].nunique())
print('TEST class counts (metadata only):')
print(TEST_ROWS['class_name'].value_counts().reindex(CLASS_NAMES).to_string())
print('Fixed TEST row-order SHA-256:', TEST_ORDER_SHA)
print('SECTION 2 PASS: Lung source-group split and TEST metadata verified; no TEST pixels read.')

Original lung manifest rows: 15000
Held-out TEST images: 2333
Reconstructed TEST source-image groups (NOT patients): 113
TEST class counts (metadata only):
class_name
lung_aca    768
lung_n      788
lung_scc    777
Fixed TEST row-order SHA-256: 210f47da525b7eda93ca9466e71c0ade94eedb02404b6a970141b3555cfb99b1
SECTION 2 PASS: Lung source-group split and TEST metadata verified; no TEST pixels read.


## Section 3: Validation-only H3 inference consistency check

Load H1 and H2 saved feature extractors, the original H3 trained head, and the exported H3 full image model using `compile=False`. Decode only three fixed **validation** examples, one from each class, directly from the already-downloaded source archive. Check the original manifest image SHA-256, reproduce exact H3 preprocessing, compare image-model and reconstructed fusion probabilities, and verify historical H3 validation predictions. **Do not use test samples to debug preprocessing.**

In [3]:
import numpy as np
import zlib
from collections import defaultdict

if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Select a Colab GPU runtime before evaluating frozen H3.')
keras.mixed_precision.set_global_policy('float32')
IMG_SIZE = 224
BATCH_SIZE = 16
h1_saved = keras.models.load_model(str(H1_MODEL), compile=False)
h2_saved = keras.models.load_model(str(H2_MODEL), compile=False)
h3_head = keras.models.load_model(str(H3_HEAD), compile=False)
h3_full = keras.models.load_model(str(H3_MODEL), compile=False)

h1_backbone = h1_saved.get_layer('mobilenetv2_1.00_224')
h2_backbone = h2_saved.get_layer('efficientnetb0')
h1_rescale = h1_saved.get_layer('mobilenetv2_rescaling')
assert isinstance(h1_rescale, keras.layers.Rescaling)
assert abs(float(h1_rescale.scale) - 1 / 127.5) < 1e-8
assert abs(float(h1_rescale.offset) + 1) < 1e-8
assert h1_backbone.trainable is False and h2_backbone.trainable is False
assert any(isinstance(x, keras.layers.Rescaling) for x in h2_backbone.layers)
assert h3_head.get_layer('fusion_class_scores').units == 3
assert tuple(h3_full.input_shape[1:]) == (224, 224, 3)
assert tuple(h3_full.output_shape[1:]) == (3,)

h1_pool = keras.Model(h1_saved.inputs, h1_saved.get_layer('global_average_pooling').output)
h2_pool = keras.Model(h2_saved.inputs, h2_saved.get_layer('global_average_pooling').output)
assert tuple(h1_pool.output_shape[1:]) == (1280,)
assert tuple(h2_pool.output_shape[1:]) == (1280,)

# Sample choice is fixed by manifest metadata before any inference.
validation_smoke = (VAL_ROWS.sort_values('source_path')
                    .groupby('class_id', sort=True).head(1)
                    .sort_values('class_id').reset_index(drop=True))
assert len(validation_smoke) == 3
assert validation_smoke['class_id'].tolist() == [0, 1, 2]

def image_member_is_safe(info):
    name = PurePosixPath(info.filename.replace('\\', '/'))
    return (not info.is_dir() and not name.is_absolute()
            and '..' not in name.parts and '__MACOSX' not in name.parts
            and not name.name.startswith('._')
            and name.suffix.lower() in ('.jpg', '.jpeg', '.png')
            and not stat.S_ISLNK(info.external_attr >> 16)
            and 0 < info.file_size <= 30 * 1024**2)

def archive_index(archive):
    by_name = defaultdict(list)
    for info in archive.infolist():
        if not image_member_is_safe(info):
            continue
        member = PurePosixPath(info.filename.replace('\\', '/'))
        classes = [part for part in member.parts if part in EXPECTED_CLASSES]
        if len(classes) == 1:
            by_name[(classes[0], member.name)].append(info)
    return by_name

def resolve_member(frame, by_name):
    result = []
    for row in frame.itertuples(index=False):
        basename = PurePosixPath(row.source_path.replace('\\', '/')).name
        matching = by_name.get((row.class_name, basename), [])
        if len(matching) != 1:
            raise RuntimeError('Expected one ZIP member for ' + row.class_name + '/' + basename
                               + ', found ' + str(len(matching)))
        info = matching[0]
        if not image_member_is_safe(info):
            raise RuntimeError('Unsafe archive image entry.')
        result.append(info)
    if len(set(info.filename for info in result)) != len(result):
        raise RuntimeError('Duplicate manifest-to-archive member resolution.')
    return result

def decode_like_h3(image_bytes):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.cast(image, tf.float32)
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE,
                                     method='bilinear', antialias=True)
    return tf.ensure_shape(image, (224, 224, 3))

validation_images = []
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    member_index = archive_index(archive)
    validation_members = resolve_member(validation_smoke, member_index)
    for row, info in zip(validation_smoke.itertuples(index=False), validation_members):
        payload = archive.read(info)
        if len(payload) != info.file_size or (zlib.crc32(payload) & 0xffffffff) != info.CRC:
            raise RuntimeError('Validation ZIP CRC mismatch: ' + info.filename)
        if hashlib.sha256(payload).hexdigest() != row.recorded_sha256:
            raise RuntimeError('Validation image SHA-256 mismatch: ' + row.source_path)
        validation_images.append(decode_like_h3(payload))
validation_batch = tf.stack(validation_images)
assert tuple(validation_batch.shape) == (3, 224, 224, 3)
assert 0 <= float(tf.reduce_min(validation_batch))
assert float(tf.reduce_max(validation_batch)) <= 255

vec1 = h1_pool(validation_batch, training=False).numpy()
vec2 = h2_pool(validation_batch, training=False).numpy()
features = np.concatenate([vec1, vec2], axis=1)
assert features.shape == (3, 2560)
reconstructed = h3_head(features, training=False).numpy()
full_probs = h3_full(validation_batch, training=False).numpy()
np.testing.assert_allclose(reconstructed, full_probs, rtol=2e-4, atol=2e-4)
assert full_probs.shape == (3, 3)
assert np.isfinite(full_probs).all()
assert np.allclose(full_probs.sum(axis=1), 1.0, atol=2e-5)

historical = pd.read_csv(H3_VAL).set_index('source_path', verify_integrity=True)
max_historical_difference = 0.0
for i, row in validation_smoke.iterrows():
    old = historical.loc[row['source_path']]
    if int(np.argmax(full_probs[i])) != int(old['predicted_id']):
        raise RuntimeError('Frozen H3 prediction disagrees with historical validation data.')
    historical_probs = np.asarray([float(old['score_' + name]) for name in CLASS_NAMES])
    max_historical_difference = max(
        max_historical_difference, float(np.abs(full_probs[i] - historical_probs).max()))
if max_historical_difference > 1e-3:
    raise RuntimeError('Saved H3 inference probabilities changed on validation examples.')

print('Frozen model:', h3_full.name)
print('Validation smoke examples:', len(validation_smoke))
print('H1/H2 pooled feature shapes:', vec1.shape, vec2.shape)
print('Concatenated feature shape:', features.shape)
print('Full H3 vs reconstructed maximum difference:',
      float(np.max(np.abs(full_probs - reconstructed))))
print('Historical H3 validation predictions matched: 3 / 3')
print('Historical validation score maximum difference:', max_historical_difference)
print('SECTION 3 PASS: Frozen H3 inference reproduced on VALIDATION only. TEST pixels untouched.')

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 6 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Frozen model: H3_Deep_Feature_Fusion_Image_Model
Validation smoke examples: 3
H1/H2 pooled feature shapes: (3, 1280) (3, 1280)
Concatenated feature shape: (3, 2560)
Full H3 vs reconstructed maximum difference: 0.0
Historical H3 validation predictions matched: 3 / 3
Historical validation score maximum difference: 1.4404296644343617e-09
SECTION 3 PASS: Frozen H3 inference reproduced on VALIDATION only. TEST pixels untouched.


## Section 4: Resolve TEST archive member inventory and record a test-once protocol

Inspect ZIP **file names and member sizes only**. Resolve all 2,333 held-out manifest rows to unique class-aware LC25000 ZIP members, preserving manifest order. Do not read member contents. Save an immutable protocol under `lung/selection/` before test evaluation, including hashes, split unit, preprocessing, output destination, one-pass policy, and predefined metrics. If a matching protocol already exists, verify it read-only.

In [4]:
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    member_index = archive_index(archive)
    test_infos = resolve_member(TEST_ROWS, member_index)
    test_members = [info.filename for info in test_infos]
    total_raw_bytes = sum(info.file_size for info in test_infos)
assert len(test_members) == 2333 and len(set(test_members)) == 2333

protocol = {
    'project': 'Stain-Robust Explainable Deep Feature Fusion Framework for Lung Cancer Histopathology Subtype Classification',
    'experiment': RUN_NAME,
    'recorded_utc': datetime.now(timezone.utc).isoformat(),
    'selection_freeze_relative_path': str(FREEZE_PATH.relative_to(ROOT)),
    'selection_freeze_sha256': sha256_file(FREEZE_PATH),
    'selected_model': 'H3_MobileNetV2_EfficientNetB0_fusion_seed42',
    'checkpoint': {'relative_path': str(H3_MODEL.relative_to(ROOT)),
                   'sha256': sha256_file(H3_MODEL)},
    'head_sha256': sha256_file(H3_HEAD),
    'H1_model_sha256': sha256_file(H1_MODEL),
    'H2_model_sha256': sha256_file(H2_MODEL),
    'configuration_sha256': sha256_file(H3_CONFIG),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'class_mapping_sha256': sha256_file(MAPPING_PATH),
    'audit_sha256': sha256_file(AUDIT_PATH),
    'archive': {'path_relative_to_root': str(ZIP_PATH.relative_to(ROOT)),
                'expected_sha256': EXPECTED_ARCHIVE_SHA256},
    'split': 'test',
    'split_unit': 'reconstructed source-image group; NOT patient/slide/hospital',
    'test_images': 2333,
    'test_source_groups': 113,
    'test_order_sha256': TEST_ORDER_SHA,
    'archive_member_order_sha256': hashlib.sha256(
        '\n'.join(test_members).encode('utf-8')).hexdigest(),
    'class_mapping': class_to_id,
    'preprocessing': ('tf.io.decode_image RGB, float32 [0,255]; '
                      'tf.image.resize_with_pad(224,224, bilinear, antialias=True); '
                      'original H3 full image model applies each backbone preprocessing; '
                      'NO Reinhard normalization, augmentation, or external rescaling'),
    'batch_size': BATCH_SIZE,
    'inference_mode': 'h3_full(x, training=False)',
    'metrics_predeclared': ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1',
                            'per_class_precision_recall_f1_support', 'confusion_matrix'],
    'no_training': True,
    'no_test_based_tuning_or_model_reselection': True,
    'one_planned_test_evaluation': True,
    'results_relative_path': str(RESULT_DIR.relative_to(ROOT)),
    'status': 'PLANNED_NOT_EVALUATED',
    'notes': ('Model selected after validation comparison, before TEST. Not preregistered. '
              'LC25000 source-group split does not establish patient or external-site independence.'),
}
PROTOCOL_PATH.parent.mkdir(parents=True, exist_ok=True)
if PROTOCOL_PATH.is_file():
    previous = read_json(PROTOCOL_PATH)
    for key, value in protocol.items():
        if key == 'recorded_utc':
            continue
        if previous.get(key) != value:
            raise RuntimeError('Existing TEST protocol differs: ' + key)
    protocol = previous
    print('Existing immutable TEST protocol verified; not overwritten.')
else:
    if TEST_RUN_STATE != 'NOT_STARTED':
        raise RuntimeError('Existing TEST inference lacks a saved protocol; stop.')
    with PROTOCOL_PATH.open('x', encoding='utf-8') as stream:
        json.dump(protocol, stream, indent=2)
        stream.write('\n')
assert read_json(PROTOCOL_PATH) == protocol
print('Resolved TEST ZIP members (metadata only):', len(test_members))
print('TEST ZIP uncompressed member size:', round(total_raw_bytes / 1024**3, 3), 'GiB')
print('Protocol:', PROTOCOL_PATH.relative_to(ROOT))
print('Output directory reserved (not yet created):', RESULT_DIR.relative_to(ROOT))
print('SECTION 4 PASS: TEST metadata and predeclared protocol verified; test pixels untouched.')

Resolved TEST ZIP members (metadata only): 2333
TEST ZIP uncompressed member size: 0.134 GiB
Protocol: lung/selection/H3_final_test_protocol_2026-10-19.json
Output directory reserved (not yet created): lung/results/H_Final_Test_H3_seed42
SECTION 4 PASS: TEST metadata and predeclared protocol verified; test pixels untouched.


## Section 5: Explicit authorization for one final H3 TEST pass

**Stop and review Section 4 before proceeding.** Enter the exact authorization phrase only when you are ready for the irreversible held-out test evaluation. This cell validates the saved freeze and protocol but does not read test pixels. If previous raw test predictions exist, do not authorize another pass.

In [5]:
if not PROTOCOL_PATH.is_file() or read_json(PROTOCOL_PATH) != protocol:
    raise RuntimeError('The saved TEST protocol is missing or changed.')
assert sha256_file(FREEZE_PATH) == protocol['selection_freeze_sha256']
assert sha256_file(H3_MODEL) == protocol['checkpoint']['sha256']
assert sha256_file(H3_HEAD) == protocol['head_sha256']
assert sha256_file(MANIFEST_PATH) == protocol['manifest_sha256']
assert TEST_ORDER_SHA == protocol['test_order_sha256']

if RESULT_DIR.exists():
    if TEST_RUN_STATE not in ('INFERENCE_COMPLETED', 'FINALIZED'):
        raise FileExistsError('A previous TEST attempt exists; do not authorize repeated inference.')
    print('SECTION 5 SKIPPED: Completed TEST inference already exists.')
    print('Do not run Section 6. Continue with saved-report review or Section 8.')
else:
    CONFIRM_PHRASE = 'AUTHORIZE H3 FINAL TEST ONCE'
    entered = input('For exactly ONE frozen H3 held-out TEST pass, type '
                    + repr(CONFIRM_PHRASE) + ': ')
    if entered != CONFIRM_PHRASE:
        raise PermissionError('Final TEST is not authorized. No test images accessed.')
    AUTHORIZATION = {
        'authorized_at_utc': datetime.now(timezone.utc).isoformat(),
        'exact_phrase_confirmed': True,
        'protocol_sha256': sha256_file(PROTOCOL_PATH),
        'freeze_sha256': sha256_file(FREEZE_PATH),
        'model_sha256': sha256_file(H3_MODEL),
        'test_order_sha256': TEST_ORDER_SHA,
    }
    print('SECTION 5 PASS: Explicit one-time H3 TEST authorization recorded in runtime memory.')
    print('No test pixels read. Section 6 starts the one planned held-out inference.')

For exactly ONE frozen H3 held-out TEST pass, type 'AUTHORIZE H3 FINAL TEST ONCE': AUTHORIZE H3 FINAL TEST ONCE
SECTION 5 PASS: Explicit one-time H3 TEST authorization recorded in runtime memory.
No test pixels read. Section 6 starts the one planned held-out inference.


## Section 6: ONE frozen H3 held-out TEST inference (test pixels first accessed here)

After authorization, verify an existing Colab-local copy of LC25000 or copy it from Drive (not a new download); require the original **archive SHA-256**. Create the permanent Drive run directory and a started receipt **before decoding the first test image**. Stream the fixed 2,333 test records in original manifest order, verify each image against the saved manifest SHA-256, decode with original H3 preprocessing, and score **only the frozen full image model** with `training=False`.

Each batch is appended to a Drive CSV containing row index, original image SHA-256, true class ID, predicted class ID, and three softmax scores. Original filenames and source-group IDs are not included in this prediction export. Save an inference-complete receipt after all 2,333 rows. No training, transformations, comparison-driven reranking, or calibration. If interrupted, **stop and preserve the partial results: do not casually rerun this cell.**

In [6]:
import csv
from tqdm.auto import tqdm

if 'AUTHORIZATION' not in globals() or not AUTHORIZATION.get('exact_phrase_confirmed'):
    raise PermissionError('Section 5 one-time authorization is required.')
if RESULT_DIR.exists():
    raise FileExistsError('A TEST attempt already exists; Section 6 must not be repeated.')
if AUTHORIZATION['protocol_sha256'] != sha256_file(PROTOCOL_PATH):
    raise RuntimeError('The saved protocol changed after authorization.')
if AUTHORIZATION['freeze_sha256'] != sha256_file(FREEZE_PATH):
    raise RuntimeError('The model-selection freeze changed after authorization.')
if AUTHORIZATION['model_sha256'] != sha256_file(H3_MODEL):
    raise RuntimeError('The H3 checkpoint changed after authorization.')
if sha256_file(H3_HEAD) != protocol['head_sha256']:
    raise RuntimeError('The H3 head changed after authorization.')
if sha256_file(H1_MODEL) != protocol['H1_model_sha256']:
    raise RuntimeError('The H1 checkpoint changed after authorization.')
if sha256_file(H2_MODEL) != protocol['H2_model_sha256']:
    raise RuntimeError('The H2 checkpoint changed after authorization.')
if sha256_file(MANIFEST_PATH) != protocol['manifest_sha256']:
    raise RuntimeError('The fixed lung manifest changed after authorization.')
if TEST_ORDER_SHA != protocol['test_order_sha256'] or len(TEST_ROWS) != 2333:
    raise RuntimeError('Fixed held-out test order/size changed after authorization.')
if hashlib.sha256('\n'.join(test_members).encode('utf-8')).hexdigest() != protocol['archive_member_order_sha256']:
    raise RuntimeError('Archive member order changed after authorization.')

# Reuse only a verified local source archive, or make a fresh local copy of the Drive ZIP.
CACHED_ZIP = Path('/content/ictiu_data/lung_h1/LC25000.zip')
PRIVATE_ZIP = Path('/content/H3_final_test_LC25000.zip')
if CACHED_ZIP.is_file():
    LOCAL_ZIP = CACHED_ZIP
    print('Checking the existing Colab-local LC25000 archive...')
else:
    LOCAL_ZIP = PRIVATE_ZIP
    if not LOCAL_ZIP.is_file():
        if shutil.disk_usage('/content').free < ZIP_PATH.stat().st_size + 2 * 1024**3:
            raise RuntimeError('Not enough temporary Colab disk space for the archive copy.')
        partial = Path(str(LOCAL_ZIP) + '.partial')
        if partial.exists():
            raise FileExistsError('Partial local archive copy exists; inspect before retrying.')
        print('Copying the original saved Drive archive to Colab disk (no download)...')
        shutil.copyfile(ZIP_PATH, partial)
        if partial.stat().st_size != ZIP_PATH.stat().st_size:
            raise RuntimeError('Incomplete local archive copy. Stop.')
        partial.replace(LOCAL_ZIP)

LOCAL_ARCHIVE_SHA = sha256_file(LOCAL_ZIP)
if LOCAL_ARCHIVE_SHA != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError('Local LC25000 ZIP SHA-256 differs from original saved dataset. STOP.')
if not zipfile.is_zipfile(LOCAL_ZIP):
    raise zipfile.BadZipFile('Verified-hash source is not readable as ZIP.')
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    resolved_local = resolve_member(TEST_ROWS, archive_index(archive))
    if [i.filename for i in resolved_local] != test_members:
        raise RuntimeError('Local test ZIP member order is not the frozen protocol order.')

# Permanent attempt guard exists before any test image is decoded.
RESULT_DIR.mkdir(parents=True, exist_ok=False)
started = {
    'status': 'TEST_INFERENCE_STARTED',
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'authorization_utc': AUTHORIZATION['authorized_at_utc'],
    'protocol_sha256': sha256_file(PROTOCOL_PATH),
    'freeze_sha256': sha256_file(FREEZE_PATH),
    'checkpoint_sha256': sha256_file(H3_MODEL),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'test_order_sha256': TEST_ORDER_SHA,
    'test_images_expected': 2333,
    'archive_sha256': LOCAL_ARCHIVE_SHA,
    'training_performed': False,
    'test_based_tuning_allowed': False,
}
with (RESULT_DIR / 'test_evaluation_started.json').open('x', encoding='utf-8') as f:
    json.dump(started, f, indent=2)
    f.write('\n')

RAW_PRED_PATH = RESULT_DIR / 'test_predictions.csv'
prob_cols = [f'prob_{j}' for j in range(3)]
columns = ['test_row_index', 'image_sha256', 'true_id', 'predicted_id'] + prob_cols
with RAW_PRED_PATH.open('x', encoding='utf-8', newline='') as out_csv:
    csv.DictWriter(out_csv, fieldnames=columns).writeheader()

n_saved = 0
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for start in tqdm(range(0, len(TEST_ROWS), BATCH_SIZE), desc='Frozen H3: one TEST pass'):
        batch_rows = TEST_ROWS.iloc[start:start + BATCH_SIZE]
        batch_members = test_members[start:start + len(batch_rows)]
        images = []
        for row, member_name in zip(batch_rows.itertuples(index=False), batch_members):
            info = archive.getinfo(member_name)
            if not image_member_is_safe(info):
                raise RuntimeError('Unsafe TEST ZIP entry. STOP.')
            payload = archive.read(info)
            if len(payload) != info.file_size or (zlib.crc32(payload) & 0xffffffff) != info.CRC:
                raise RuntimeError('TEST image failed ZIP size/CRC check. STOP.')
            if hashlib.sha256(payload).hexdigest() != row.recorded_sha256:
                raise RuntimeError('TEST image differs from fixed manifest SHA-256. STOP.')
            images.append(decode_like_h3(payload))
        batch = tf.stack(images, axis=0)
        if tuple(batch.shape[1:]) != (224, 224, 3):
            raise RuntimeError('Incorrect TEST input shape.')
        probs = h3_full(batch, training=False).numpy()
        if probs.shape != (len(batch_rows), 3) or not np.isfinite(probs).all():
            raise RuntimeError('Invalid H3 TEST output shape or non-finite score.')
        if (probs < -1e-6).any() or (probs > 1.000001).any():
            raise RuntimeError('Invalid TEST softmax probability range.')
        if not np.allclose(probs.sum(axis=1), 1.0, atol=2e-5):
            raise RuntimeError('TEST softmax rows do not sum to one.')
        predicted = np.argmax(probs, axis=1)
        records = []
        for j, row in enumerate(batch_rows.itertuples(index=False)):
            item = {
                'test_row_index': int(start + j),
                'image_sha256': row.recorded_sha256,
                'true_id': int(row.class_id),
                'predicted_id': int(predicted[j]),
            }
            item.update({f'prob_{k}': float(probs[j, k]) for k in range(3)})
            records.append(item)
        with RAW_PRED_PATH.open('a', encoding='utf-8', newline='') as out_csv:
            csv.DictWriter(out_csv, fieldnames=columns).writerows(records)
        n_saved += len(records)

if n_saved != 2333:
    raise RuntimeError(f'Incomplete TEST inference: {n_saved} of 2333 images saved.')
completed = {
    'status': 'TEST_INFERENCE_COMPLETED',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'test_images_scored': n_saved,
    'test_source_groups': 113,
    'predictions_sha256': sha256_file(RAW_PRED_PATH),
    'test_order_sha256': TEST_ORDER_SHA,
    'checkpoint_sha256': sha256_file(H3_MODEL),
    'no_training': True,
    'no_test_based_tuning': True,
}
with (RESULT_DIR / 'test_inference_completed.json').open('x', encoding='utf-8') as f:
    json.dump(completed, f, indent=2)
    f.write('\n')
print('Saved raw TEST predictions:', RAW_PRED_PATH)
print('Scored TEST images:', n_saved)
print('TEST inference checkpoint:', RESULT_DIR / 'test_inference_completed.json')
print('SECTION 6 PASS: Exactly one frozen H3 held-out TEST inference completed and saved.')

Copying the original saved Drive archive to Colab disk (no download)...


Frozen H3: one TEST pass:   0%|          | 0/146 [00:00<?, ?it/s]

Saved raw TEST predictions: /content/drive/MyDrive/ICTIU_Final_Projects/lung/results/H_Final_Test_H3_seed42/test_predictions.csv
Scored TEST images: 2333
TEST inference checkpoint: /content/drive/MyDrive/ICTIU_Final_Projects/lung/results/H_Final_Test_H3_seed42/test_inference_completed.json
SECTION 6 PASS: Exactly one frozen H3 held-out TEST inference completed and saved.


## Section 7: Generate final TEST metrics and immutable report evidence (no new inference)

Use the saved predictions CSV only, not the model or dataset ZIP. Verify receipt checksums, exact manifest order, saved per-image SHA-256, true IDs, predicted argmax and probability validity. Compute the originally specified metrics and class confusion matrix, then export a report summary, figures, environment, provenance and artifact hashes in a new lung-only result folder. Do not revisit model choice or tune H3 after viewing these results.

In [7]:
import importlib.metadata
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

STARTED_PATH = RESULT_DIR / 'test_evaluation_started.json'
COMPLETE_PATH = RESULT_DIR / 'test_inference_completed.json'
FINAL_MARKER = RESULT_DIR / 'FINALIZED.json'
if FINAL_MARKER.exists():
    raise FileExistsError('Final TEST report already exists. Use Section 8, not repeated export.')
if not (STARTED_PATH.is_file() and COMPLETE_PATH.is_file() and RAW_PRED_PATH.is_file()):
    raise RuntimeError('The one-time TEST inference has not completed. Stop.')
started = read_json(STARTED_PATH)
completed = read_json(COMPLETE_PATH)
assert started['status'] == 'TEST_INFERENCE_STARTED'
assert completed['status'] == 'TEST_INFERENCE_COMPLETED'
assert completed['test_images_scored'] == 2333
assert completed['test_order_sha256'] == TEST_ORDER_SHA
assert started['protocol_sha256'] == sha256_file(PROTOCOL_PATH)
assert started['freeze_sha256'] == sha256_file(FREEZE_PATH)
assert started['checkpoint_sha256'] == sha256_file(H3_MODEL)
assert started['archive_sha256'] == EXPECTED_ARCHIVE_SHA256
assert sha256_file(RAW_PRED_PATH) == completed['predictions_sha256']

pred_table = pd.read_csv(RAW_PRED_PATH, dtype={'image_sha256': str})
assert len(pred_table) == 2333
assert pred_table['test_row_index'].tolist() == list(range(2333))
assert pred_table['image_sha256'].tolist() == TEST_ROWS['recorded_sha256'].tolist()
assert pred_table['true_id'].tolist() == TEST_ROWS['class_id'].tolist()
assert pred_table['predicted_id'].isin([0, 1, 2]).all()
scores = pred_table[[f'prob_{k}' for k in range(3)]].to_numpy(dtype=np.float64)
assert scores.shape == (2333, 3) and np.isfinite(scores).all()
assert ((scores >= -1e-6) & (scores <= 1.000001)).all()
assert np.allclose(scores.sum(axis=1), 1.0, atol=2e-5)
assert pred_table['predicted_id'].tolist() == np.argmax(scores, axis=1).tolist()

y_true = pred_table['true_id'].to_numpy(dtype=np.int64)
y_pred = pred_table['predicted_id'].to_numpy(dtype=np.int64)
labels = [0, 1, 2]
accuracy = float(accuracy_score(y_true, y_pred))
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average='macro', zero_division=0)
per_p, per_r, per_f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average=None, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=labels)
assert cm.shape == (3, 3) and int(cm.sum()) == 2333

metrics = {
    'partition': 'TEST',
    'experiment': RUN_NAME,
    'selected_model': 'H3_MobileNetV2_EfficientNetB0_fusion_seed42',
    'n_images': 2333,
    'n_reconstructed_source_groups': 113,
    'grouping_unit': 'reconstructed source-image group; NOT patient ID',
    'accuracy': accuracy,
    'macro_precision': float(macro_p),
    'macro_recall': float(macro_r),
    'macro_f1': float(macro_f1),
    'n_correct': int((y_true == y_pred).sum()),
    'n_errors': int((y_true != y_pred).sum()),
    'source_predictions_sha256': sha256_file(RAW_PRED_PATH),
    'no_post_test_tuning': True,
}
per_class = pd.DataFrame({
    'class_id': labels,
    'class_name': CLASS_NAMES,
    'precision': per_p.astype(float),
    'recall': per_r.astype(float),
    'f1': per_f1.astype(float),
    'support': support.astype(int),
})

STAGE = Path('/content/H_Final_Test_H3_report_staging')
if STAGE.exists():
    raise FileExistsError('Local report staging already exists; inspect before rerunning: ' + str(STAGE))
STAGE.mkdir(parents=True)
(STAGE / 'test_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
per_class.to_csv(STAGE / 'test_per_class_metrics.csv', index=False)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    STAGE / 'test_confusion_matrix.csv', index_label='true_class')
report_text = classification_report(
    y_true, y_pred, labels=labels, target_names=CLASS_NAMES, digits=4, zero_division=0)
(STAGE / 'test_classification_report.txt').write_text(report_text, encoding='utf-8')
report_dict = classification_report(
    y_true, y_pred, labels=labels, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
pd.DataFrame(report_dict).transpose().to_csv(STAGE / 'test_classification_report.csv')

fig, ax = plt.subplots(figsize=(8.5, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, xticks_rotation=20, colorbar=False, values_format='d')
ax.set_title('Frozen H3 feature fusion: Final held-out TEST')
fig.tight_layout()
fig.savefig(STAGE / 'test_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.close(fig)

environment = {
    'python': platform.python_version(),
    'tensorflow': tf.__version__,
    'visible_gpus': [str(g) for g in tf.config.list_physical_devices('GPU')],
    'packages': {},
}
for package in ['keras', 'numpy', 'pandas', 'scikit-learn', 'matplotlib']:
    try:
        environment['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        environment['packages'][package] = 'not installed'
(STAGE / 'test_environment.json').write_text(
    json.dumps(environment, indent=2) + '\n', encoding='utf-8')

provenance = {
    'report_status': 'FINAL_HELD_OUT_TEST_EVALUATED_ONCE',
    'partition': 'TEST',
    'completed_at_utc': completed['completed_at_utc'],
    'report_generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'protocol_file': str(PROTOCOL_PATH.relative_to(ROOT)),
    'protocol_sha256': sha256_file(PROTOCOL_PATH),
    'freeze_file': str(FREEZE_PATH.relative_to(ROOT)),
    'freeze_sha256': sha256_file(FREEZE_PATH),
    'model_sha256': sha256_file(H3_MODEL),
    'H3_head_sha256': sha256_file(H3_HEAD),
    'H1_model_sha256': sha256_file(H1_MODEL),
    'H2_model_sha256': sha256_file(H2_MODEL),
    'configuration_sha256': sha256_file(H3_CONFIG),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'mapping_sha256': sha256_file(MAPPING_PATH),
    'archive_sha256': started['archive_sha256'],
    'test_order_sha256': TEST_ORDER_SHA,
    'predictions_sha256': sha256_file(RAW_PRED_PATH),
    'test_images': 2333,
    'test_reconstructed_source_groups': 113,
    'training_performed': False,
    'model_selection_or_tuning_on_test': False,
    'notes': ('LC25000 lung-only source-group held-out partition. '
              'Source-image grouping is not patient, slide or external-site separation.'),
}
(STAGE / 'test_run_provenance.json').write_text(
    json.dumps(provenance, indent=2) + '\n', encoding='utf-8')

summary = (
    '# Frozen H3 Deep Feature Fusion: Final Held-out TEST\n\n'
    '**Evaluation:** One planned pass on 2,333 held-out lung histopathology images '
    'from 113 reconstructed source-image groups (not patients).\n\n'
    f'- TEST accuracy: **{accuracy:.4%}**\n'
    f'- TEST macro-precision: **{macro_p:.6f}**\n'
    f'- TEST macro-recall: **{macro_r:.6f}**\n'
    f'- TEST macro-F1: **{macro_f1:.6f}**\n'
    f'- TEST correct: **{metrics["n_correct"]}/2333**\n'
    '- Model: frozen H3 concatenated pooled-feature fusion selected on validation.\n'
    '- No test-based model selection, stain-method changes, or threshold optimization.\n'
    '- Earlier H1/H2/H3/H4 comparisons are VALIDATION only.\n'
    '- No external-hospital or patient-level independence is demonstrated.\n\n'
    'Softmax scores are not calibrated clinical probabilities.\n'
)
(STAGE / 'RUN_SUMMARY.md').write_text(summary, encoding='utf-8')

for path in sorted(STAGE.iterdir()):
    dest = RESULT_DIR / path.name
    if dest.exists():
        raise FileExistsError('Existing TEST report evidence must not be overwritten: ' + str(dest))
    shutil.copy2(path, dest)
    if sha256_file(path) != sha256_file(dest):
        raise RuntimeError('Copied TEST artifact differs: ' + path.name)

with FINAL_MARKER.open('x', encoding='utf-8') as out_json:
    json.dump({'status': 'TEST_REPORT_FINALIZED',
               'created_utc': datetime.now(timezone.utc).isoformat(),
               'test_images': 2333,
               'test_predictions_sha256': sha256_file(RAW_PRED_PATH)}, out_json, indent=2)
    out_json.write('\n')

checksums = {
    str(p.relative_to(RESULT_DIR)): sha256_file(p)
    for p in sorted(RESULT_DIR.iterdir())
    if p.is_file() and p.name != 'artifact_checksums.json'
}
with (RESULT_DIR / 'artifact_checksums.json').open('x', encoding='utf-8') as out_json:
    json.dump(checksums, out_json, indent=2)
    out_json.write('\n')

print('FINAL HELD-OUT TEST: Lung H3 (2,333 images)')
print('TEST accuracy:', f'{accuracy:.4%}')
print('TEST macro-F1:', f'{macro_f1:.6f}')
print('TEST macro-precision:', f'{macro_p:.6f}')
print('TEST macro-recall:', f'{macro_r:.6f}')
print('TEST confusion matrix saved, rows=true / cols=predicted.')
print('Saved final TEST report:', RESULT_DIR)
print('SECTION 7 PASS: Final TEST metrics exported from existing predictions; no repeat inference.')

FINAL HELD-OUT TEST: Lung H3 (2,333 images)
TEST accuracy: 96.2280%
TEST macro-F1: 0.961961
TEST macro-precision: 0.962412
TEST macro-recall: 0.961992
TEST confusion matrix saved, rows=true / cols=predicted.
Saved final TEST report: /content/drive/MyDrive/ICTIU_Final_Projects/lung/results/H_Final_Test_H3_seed42
SECTION 7 PASS: Final TEST metrics exported from existing predictions; no repeat inference.


## Section 8: Read-only final integrity and provenance verification

Verify saved artifacts, checksums, frozen checkpoint hashes, manifest order, the 2,333-row inference receipt, and final report metadata. This section does not read ZIP image pixels, call the model or run another test evaluation. Save the executed notebook with outputs after completing this section.

In [8]:
if not (RESULT_DIR / 'FINALIZED.json').is_file():
    raise RuntimeError('Final TEST report is not complete. Stop and inspect the saved run.')
checksums = read_json(RESULT_DIR / 'artifact_checksums.json')
for rel, digest in checksums.items():
    if sha256_file(RESULT_DIR / rel) != digest:
        raise RuntimeError('Saved TEST artifact checksum differs: ' + rel)

saved_metrics = read_json(RESULT_DIR / 'test_metrics.json')
saved_provenance = read_json(RESULT_DIR / 'test_run_provenance.json')
saved_final = read_json(RESULT_DIR / 'FINALIZED.json')
assert saved_metrics['partition'] == 'TEST'
assert saved_metrics['n_images'] == saved_final['test_images'] == 2333
assert saved_metrics['n_reconstructed_source_groups'] == 113
assert saved_provenance['report_status'] == 'FINAL_HELD_OUT_TEST_EVALUATED_ONCE'
assert saved_provenance['training_performed'] is False
assert saved_provenance['model_selection_or_tuning_on_test'] is False
assert sha256_file(H3_MODEL) == freeze['model_artifact']['sha256']
assert sha256_file(H3_HEAD) == freeze['head_artifact']['sha256']
assert sha256_file(H1_MODEL) == EXPECTED_SHA['H1_model']
assert sha256_file(H2_MODEL) == EXPECTED_SHA['H2_model']
assert sha256_file(MANIFEST_PATH) == freeze['manifest']['sha256']
assert sha256_file(MAPPING_PATH) == freeze['mapping_file']['sha256']
assert sha256_file(FREEZE_PATH) == saved_provenance['freeze_sha256']
assert sha256_file(RESULT_DIR / 'test_predictions.csv') == saved_metrics['source_predictions_sha256']

print('Verified saved TEST artifacts:', len(checksums))
print('Frozen H3 full-image checkpoint unchanged: True')
print('H3 head and H1/H2 reference checkpoints unchanged: True')
print('Reconstructed source-group held-out TEST images:', saved_metrics['n_images'])
print('TEST accuracy:', f"{saved_metrics['accuracy']:.4%}")
print('TEST macro-F1:', f"{saved_metrics['macro_f1']:.6f}")
print('Retuning based on TEST results: NOT PERMITTED')
print('SECTION 8 PASS: Lung H3 final held-out TEST report and provenance verified.')

Verified saved TEST artifacts: 13
Frozen H3 full-image checkpoint unchanged: True
H3 head and H1/H2 reference checkpoints unchanged: True
Reconstructed source-group held-out TEST images: 2333
TEST accuracy: 96.2280%
TEST macro-F1: 0.961961
Retuning based on TEST results: NOT PERMITTED
SECTION 8 PASS: Lung H3 final held-out TEST report and provenance verified.


## Reporting, reproducibility and project separation

- Use the **final H3 TEST** metrics only after completing Section 7 and confirming Section 8. Record TEST results separately from H1/H2/H3/H4 **validation** tables.
- Do not say the 2,333 held-out lung images represent unseen patients or an outside hospital. Reconstructed source-image groups reduce augmented-copy leakage but are not clinical or institutional independence.
- H3 uses pooled-feature concatenation from frozen H1/H2 branches. H4-N and H4-NA are separate **validation-only stain-processing ablations**, not part of this selected H3 inference pipeline. No Reinhard normalization should be added here.
- Lung H3 Grad-CAM results are from fixed **validation** images; they are not a test measurement or ground-truth tumor localization.
- Do not select, retune, recalibrate or repeatedly evaluate H3 after viewing TEST outcomes. If a genuine procedural defect appears, preserve evidence and document a controlled investigation rather than silently repeating the held-out test.
- For the independent lung repository, retain the original selected source notebooks and training provenance. Do not publish patient-sensitive details, large raw image archives, private Drive files, credentials, or `.keras` weights without confirming reuse rights.
- Reports should describe frozen checkpoint selection on validation, final one-time TEST evaluation, class-wise errors, and limitations honestly. Save this notebook **after actual Colab execution** and retain its outputs.